# Practical 7: Artistic Content Generation using Neural Style Transfer and GAN

## Aim
To generate artistic content using neural style transfer and GAN-based models with PyTorch.

In [ ]:
%pip install -q torch torchvision pillow matplotlib requests

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from PIL import Image
import requests
from io import BytesIO
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

## Part A: Neural Style Transfer

In [ ]:
def load_image_from_url(url, size=256):
    response = requests.get(url, timeout=20)
    img = Image.open(BytesIO(response.content)).convert('RGB')
    tfm = transforms.Compose([
        transforms.Resize((size, size)),
        transforms.ToTensor()
    ])
    return tfm(img).unsqueeze(0).to(device)

content_url = 'https://upload.wikimedia.org/wikipedia/commons/0/00/Tuebingen_Neckarfront.jpg'
style_url = 'https://upload.wikimedia.org/wikipedia/commons/6/60/Vincent_van_Gogh_-_Starry_Night_-_Google_Art_Project.jpg'

content = load_image_from_url(content_url, size=256)
style = load_image_from_url(style_url, size=256)

print('Content and style images loaded.')

In [ ]:
vgg = models.vgg19(weights=models.VGG19_Weights.DEFAULT).features.to(device).eval()
for param in vgg.parameters():
    param.requires_grad = False

feature_layers = {'0': 'conv1_1', '5': 'conv2_1', '10': 'conv3_1', '19': 'conv4_1', '28': 'conv5_1'}

def get_features(x):
    feats = {}
    for name, layer in vgg._modules.items():
        x = layer(x)
        if name in feature_layers:
            feats[feature_layers[name]] = x
    return feats

def gram_matrix(tensor):
    b, c, h, w = tensor.size()
    x = tensor.view(c, h * w)
    return x @ x.t()

style_features = get_features(style)
style_grams = {k: gram_matrix(v) for k, v in style_features.items()}

generated = content.clone().requires_grad_(True)
optimizer = optim.Adam([generated], lr=0.02)

style_weight = 1e6
content_weight = 1e0

for step in range(1, 201):
    gen_features = get_features(generated)
    content_loss = torch.mean((gen_features['conv4_1'] - get_features(content)['conv4_1']) ** 2)

    style_loss = 0
    for layer in style_grams:
        gen_gram = gram_matrix(gen_features[layer])
        target_gram = style_grams[layer]
        _, c, h, w = gen_features[layer].shape
        style_loss += torch.mean((gen_gram - target_gram) ** 2) / (c * h * w)

    loss = content_weight * content_loss + style_weight * style_loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    with torch.no_grad():
        generated.clamp_(0, 1)

    if step % 50 == 0:
        print(f'Step {step}: Total Loss = {loss.item():.4f}')

print('Neural style transfer completed.')

In [ ]:
def show_tensor_image(t, title):
    img = t.detach().cpu().squeeze(0).permute(1, 2, 0).numpy()
    plt.imshow(img)
    plt.title(title)
    plt.axis('off')

plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
show_tensor_image(content, 'Content Image')
plt.subplot(1, 3, 2)
show_tensor_image(style, 'Style Image')
plt.subplot(1, 3, 3)
show_tensor_image(generated, 'Stylized Output')
plt.tight_layout()
plt.show()

## Part B: GAN-based Artistic Generation (Optional)
Use the GAN model from Practical 5 to generate abstract grayscale artistic textures by sampling different random latent vectors.

In [ ]:
print('Optional GAN section: Reuse trained Generator from Practical 5 for artistic samples.')
print('You can import the saved generator weights and create new art-like variations by latent interpolation.')

## Result
Artistic content was generated using neural style transfer, and GAN-based artistic generation workflow was studied using PyTorch.